In [21]:
import pandas as pd
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config; set_config(display='diagram')

from xgboost import XGBClassifier
import sys
from pathlib import Path
# Step up to the root folder so Python can see 'custom_transformers'
sys.path.append(str(Path.cwd().parent))

In [22]:
def load_data(file_path: str):
    """
    Loads a dataset
    """
    df = pd.read_csv(file_path)
    return df

In [23]:
def load_trained_artifacts():
    """
    Loads saved processor, model and pipeline

    """
    BASE_PATH = '../models/'
    preproc_path = BASE_PATH +'xgb_churn_preproc_pipeline.pkl'
    model_path = BASE_PATH + 'xgb_churn_model.pkl'
    full_pipeline_path = BASE_PATH +'xgb_churn_pipeline_with_segmentation_with_grid_search.pkl'

    #Load artifacts from memory
    print("🔄 Loading pipeline components, preprocessor and model...")
    preprocessor = joblib.load(preproc_path)
    model = joblib.load(model_path)
    print("🔄 Loading full pipeline...")
    pipeline = joblib.load(full_pipeline_path)

    print("✅ Artifacts successfully loaded!")
    return preprocessor, model, pipeline


In [24]:
def transform_columns(df: pd.DataFrame):

    clean_cols = df.columns.str.strip().str.replace(' ', '_')

    # Insert underscore between lowercase letters/digits and an uppercase letter (e.g., CustomerID -> Customer_ID)
    clean_cols = clean_cols.str.replace(r'([a-z0-9])([A-Z])', r'\1_\2', regex=True)

    # Insert underscore between consecutive uppercase letters followed by lowercase (e.g., USAUser -> USA_User)
    clean_cols = clean_cols.str.replace(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', regex=True)

    # Lowercase everything and collapse any sequential underscores (e.g., customer__id -> customer_id)
    df.columns = clean_cols.str.lower().str.replace(r'_+', '_', regex=True)
    return df


### Load pickles from xgb_serialization_and_evaluation

In [25]:
preprocessor, model, pipeline = load_trained_artifacts()

🔄 Loading pipeline components, preprocessor and model...
🔄 Loading full pipeline...
✅ Artifacts successfully loaded!


### Load validation data from xgb_serialization_and_evaluation

In [26]:
X_val = load_data('../data/X_val_xgb.csv')
y_val = load_data('../data/y_val_xgb.csv')

In [27]:
X_val.head()

,account_age,monthly_charges,total_charges,subscription_type,payment_method,paperless_billing,content_type,multi_device_access,device_registered,viewing_hours_per_week,average_viewing_duration,content_downloads_per_month,genre_preference,user_rating,support_tickets_per_month,gender,watchlist_size,parental_control,subtitles_enabled
0,31,18.871369,585.012434,Basic,Mailed check,No,Movies,Yes,Computer,20.515996,154.972317,22,Action,2.026801,9,Female,23,No,Yes
1,34,17.339934,589.557749,Premium,Bank transfer,No,Both,No,Computer,36.016501,37.410487,27,Action,3.093885,6,Female,23,No,Yes
2,88,5.321946,468.331228,Premium,Bank transfer,No,TV Shows,Yes,TV,11.719514,22.917143,27,Action,4.792676,1,Male,20,No,No
3,37,15.836623,585.955056,Premium,Electronic check,Yes,Both,No,TV,16.745304,84.746628,18,Fantasy,4.516890,7,Male,11,Yes,Yes
4,94,8.746215,822.144229,Basic,Bank transfer,Yes,Movies,No,Computer,18.333511,159.207305,42,Fantasy,1.682190,4,Male,22,Yes,No


In [28]:
y_val.head()

,churn
0,0
1,0
2,0
3,1
4,0


### Model performance

In [29]:
def predict(X_val, y_val, pipeline):
    # 3. Evaluate the model on unseen test data
    y_pred = pipeline.predict(X_val)          # Hard classification: 0 (Stay) or 1 (Churn)
    y_proba = pipeline.predict_proba(X_val)[:, 1] # Risk probabilities (e.g., 0.84)

    print("\n================ MODEL PERFORMANCE ================")
    print(classification_report(y_val, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_val, y_proba):.4f}")
    print("============================================= =======")

In [30]:
predict(X_val, y_val, pipeline)


================ MODEL PERFORMANCE ================
              precision    recall  f1-score   support

           0       0.91      0.67      0.77     39922
           1       0.32      0.69      0.44      8836

    accuracy                           0.68     48758
   macro avg       0.61      0.68      0.60     48758
weighted avg       0.80      0.68      0.71     48758

ROC-AUC Score: 0.7466
============================================= =======
